# 01 Karnataka feed quality

Question (T106 only): how complete, duplicated and late are the Karnataka feeds we hold?
Coverage heatmap, missing-day calendar, duplicate fraction and reporting-lag ECDF, each with its
exact denominator.

Frozen inputs: the S08 IMD year files named in the raw ledger (`data/raw/ledger.jsonl`, local
only: IMD forbids reproduction, so only derived counts are exported), the committed S10 ADS-B
sample (`tests/fixtures/s10_adsb_lol_bengaluru_100nm_2026-11-09.json`) and the committed S01
model-run metadata. Limits are stated, never filled: S22/S23 prices have no rows (blocked,
T024/T025), and S01/S09 have one local capture day.

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/01_ka_quality/` (or `ATLAS_EDA_OUT`).

In [ ]:
import csv
import json
import os
from datetime import date
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from atlas.analysis.quality import (
    caption,
    duplicate_fraction,
    ecdf,
    ledger_duplicates,
    missing_days,
    month_coverage,
    s08_cell_days,
)
from atlas.features.m05 import year_artifacts
from atlas.ingest.s08 import URL

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/01_ka_quality"))
OUT.mkdir(parents=True, exist_ok=True)
START, END = date(1981, 1, 1), date(2025, 12, 31)


def write_csv(name, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)

In [ ]:
raws = year_artifacts(Path("data/raw"), range(START.year, END.year + 1))
present, cells = s08_cell_days(raws, "ka")
coverage = month_coverage(present, cells, START, END)
missing = missing_days(present, cells, START, END)
rows_dup = duplicate_fraction((d, c) for d in present for c in range(present[d]))
files_dup = ledger_duplicates(Path("data/raw/ledger.jsonl"), URL + "?rain=")
fetched = sorted(r.fetched_at for r in raws)
write_csv("s08_coverage.csv", coverage)
write_csv("s08_missing_days.csv", missing or [{"date": None, "missing_cells": 0, "cells": cells}])
expected = sum(r["expected"] for r in coverage)
print(cells, "cells;", sum(r["present"] for r in coverage), "of", expected, "cell-days present;")
print(len(missing), "missing days;", rows_dup, files_dup)

In [ ]:
sample = Path("tests/fixtures/s10_adsb_lol_bengaluru_100nm_2026-11-09.json")
adsb = json.loads(sample.read_text("utf-8"))
ages = [a["seen_pos"] for a in adsb["ac"] if "seen_pos" in a]
s10_ecdf = ecdf(ages)
write_csv("s10_position_age_ecdf.csv", s10_ecdf)
meta_file = Path("tests/fixtures/s01_openmeteo_meta_ecmwf_ifs025_2026-11-05.json")
meta = json.loads(meta_file.read_text("utf-8"))
s01_lag_s = meta["last_run_availability_time"] - meta["last_run_initialisation_time"]
print(len(ages), "of", len(adsb["ac"]), "aircraft with a position")
print("S01 run available", s01_lag_s, "s after initialisation")

In [ ]:
years = sorted({r["year"] for r in coverage})
share = {(r["year"], r["month"]): r["share"] for r in coverage}
grid = [[share[y, m] for m in range(1, 13)] for y in years]
fig, ax = plt.subplots(figsize=(7, 9))
im = ax.imshow(grid, aspect="auto", vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(12), [str(m) for m in range(1, 13)])
ax.set_yticks(range(0, len(years), 5), [str(y) for y in years[::5]])
ax.set_xlabel("Month")
ax.set_ylabel("Year")
fig.colorbar(im, label="Share of cell-days present")
cap_cov = caption(
    "S08 IMD 0.25 deg daily rainfall, Karnataka box",
    "cell-days with a value",
    f"share of calendar days x {cells} cells",
    START.isoformat(),
    END.isoformat(),
    f"{len(missing)} days with a missing cell",
)
ax.set_title("S08 coverage", fontsize=10)
fig.text(0.01, 0.005, cap_cov, fontsize=6)
fig.savefig(OUT / "s08_coverage.png", dpi=80, metadata={"Software": None})
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
ax.step([r["value"] for r in s10_ecdf], [r["share"] for r in s10_ecdf], where="post")
ax.set_xlabel("Position age at fetch (s)")
ax.set_ylabel("Share of aircraft")
cap_lag = caption(
    "S10 adsb.lol, Bengaluru 100 nm",
    "seconds since the last position report",
    "s",
    "2026-11-09",
    "2026-11-09",
    f"{len(ages)} of {len(adsb['ac'])} aircraft",
)
ax.set_title("S10 reporting lag ECDF", fontsize=10)
fig.text(0.01, 0.005, cap_lag, fontsize=6)
fig.savefig(OUT / "s10_position_age_ecdf.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
summary = {
    "s08": {
        "cells": cells,
        "days": len(present),
        "expected_cell_days": sum(r["expected"] for r in coverage),
        "present_cell_days": sum(r["present"] for r in coverage),
        "missing_days": len(missing),
        "row_duplicates": rows_dup,
        "file_duplicates": files_dup,
        "data_start": START.isoformat(),
        "data_end": END.isoformat(),
        "first_fetch": fetched[0].isoformat(),
        "last_fetch": fetched[-1].isoformat(),
        "caption": cap_cov,
    },
    "s10": {
        "aircraft": len(adsb["ac"]),
        "with_position": len(ages),
        "fetched_ms": adsb["now"],
        "max_age_s": max(ages),
        "caption": cap_lag,
    },
    "s01": {
        "runs": 1,
        "availability_lag_s": s01_lag_s,
        "initialised": meta["last_run_initialisation_time"],
        "available": meta["last_run_availability_time"],
    },
    "limits": [
        "S22/S23 prices: no rows (blocked, T024/T025; price denominator unverified).",
        "S01 and S09: one capture day (5 Nov 2026): no daily coverage or missing-day calendar.",
        "S10: one 9 Nov 2026 snapshot of 20 aircraft; the ECDF is that snapshot only.",
    ],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))